In [1]:
import pandas as pd
import numpy as np
import glob
import os
from tqdm import tqdm

In [2]:
!pwd

/storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Drosophila


In [3]:
!pwd

/storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Drosophila


In [4]:
BASE_DIR     = '/storage/Arushi/090526_EvoAge/kg_formation/'
MAPPING_DIR  = BASE_DIR + 'data_collection/databases_for_mapping/'
PROC_DIR     = BASE_DIR + 'processed_data/'

# ── Output path ───────────────────────────────────────────────────────────────
!mkdir Drosophila_gene_inhibits_biologicalprocess
OUT_PATH = BASE_DIR + 'processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Drosophila/Drosophila_gene_inhibits_biologicalprocess/Drosophila_gene_inhibits_biologicalprocess.csv'

# ── Required output schema ────────────────────────────────────────────────────
REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'kg_type',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'species'
]

In [5]:
#

# Genage

In [6]:
Genage = pd.read_csv(PROC_DIR + 'genage/Droso/Droso_Gene_Inhibits_BiologicalProcess.csv')
Genage.columns = Genage.columns.str.lower()
Genage = Genage[~Genage['head_detail_name'].isna()]

Genage['kg_type'] = 'Aging'

print(f"Genage: {len(Genage):,} rows")
Genage['head_id_is'] = 'NCBI_ID'
Genage['tail_id_is'] = 'Quick_GO'
Genage['species'] = 'Drosophila melanogaster'
Genage['kg_source'] = 'Genage'

Genage

Genage: 107 rows


,head,relation,tail,head_type,tail_type,head_detail_name,tail_detail_name,relation_source,species,data_type,head_alt_name,kg_type,head_id_is,tail_id_is,kg_source
0,Atg7,Gene_Inhibits_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,Autophagy-related 7,obsolete aging,GeneAge,Drosophila melanogaster,Causative,Atg7,Aging,NCBI_ID,Quick_GO,Genage
1,Atg8a,Gene_Inhibits_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,Autophagy-related 8a,obsolete aging,GeneAge,Drosophila melanogaster,Causative,Atg8a,Aging,NCBI_ID,Quick_GO,Genage
2,bchs,Gene_Inhibits_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,blue cheese,obsolete aging,GeneAge,Drosophila melanogaster,Causative,bchs,Aging,NCBI_ID,Quick_GO,Genage
3,car,Gene_Inhibits_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,carnation,obsolete aging,GeneAge,Drosophila melanogaster,Causative,car,Aging,NCBI_ID,Quick_GO,Genage
4,Cat,Gene_Inhibits_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,Catalase,obsolete aging,GeneAge,Drosophila melanogaster,Causative,Cat,Aging,NCBI_ID,Quick_GO,Genage
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102,Drp1,Gene_Inhibits_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,Dynamin related protein 1,obsolete aging,GeneAge,Drosophila melanogaster,Causative,Drp1,Aging,NCBI_ID,Quick_GO,Genage
103,Pgam5,Gene_Inhibits_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,Phosphoglycerate mutase 5,obsolete aging,GeneAge,Drosophila melanogaster,Causative,Pgam5,Aging,NCBI_ID,Quick_GO,Genage
104,Prp19,Gene_Inhibits_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,Pre-RNA processing factor 19,obsolete aging,GeneAge,Drosophila melanogaster,Causative,Prp19,Aging,NCBI_ID,Quick_GO,Genage
105,Sirt4,Gene_Inhibits_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,Sirtuin 4,obsolete aging,GeneAge,Drosophila melanogaster,Causative,Sirt4,Aging,NCBI_ID,Quick_GO,Genage


# AgingBank

In [7]:
AgingBank = pd.read_csv(PROC_DIR + 'agingbank/Droso_AgingbankGene_inhibits_BioPro.csv')
AgingBank.columns = AgingBank.columns.str.lower()
AgingBank = AgingBank[~AgingBank['head_detail_name'].isna()]

AgingBank['tail_type'] = 'BiologicalProcess'
AgingBank['relation'] = 'Gene_Inhibits_BiologicalProcess'
print(f"AgingBank: {len(AgingBank):,} rows")
AgingBank['head_id_is'] = 'NCBI_ID'
AgingBank['tail_id_is'] = 'Quick_GO'
AgingBank['species'] = 'Drosophila melanogaster'
AgingBank['kg_source'] = 'AgingBank'
AgingBank

AgingBank: 2 rows


,head,relation,tail,head_type,tail_type,kg_source,kg_type,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_type.1,species
0,SOD1,Gene_Inhibits_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,AgingBank,Aging,"superoxide dismutase 1, soluble",aging,NCBI_ID,Quick_GO,Aging,Drosophila melanogaster
1,STAC,Gene_Inhibits_BiologicalProcess,GO:0007568,Gene,BiologicalProcess,AgingBank,Aging,SH3 and cysteine rich domain,aging,NCBI_ID,Quick_GO,Aging,Drosophila melanogaster


# Consolidate into Unified Schem

In [8]:
# List all source DataFrames to include
source_dfs = [
    Genage, AgingBank
    
]

aligned = []
for df in source_dfs:
    df = df.copy()
    for col in REQUIRED_COLS:
        if col not in df.columns:
            df[col] = None       
    aligned.append(df[REQUIRED_COLS])

final_df = pd.concat(aligned, ignore_index=True)
print(f"Consolidated rows: {len(final_df):,}")
final_df

Consolidated rows: 109


,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species
0,Atg7,Gene_Inhibits_BiologicalProcess,GO:0007568,Gene,None,BiologicalProcess,Genage,Aging,NCBI_ID,Quick_GO,Autophagy-related 7,obsolete aging,Drosophila melanogaster
1,Atg8a,Gene_Inhibits_BiologicalProcess,GO:0007568,Gene,None,BiologicalProcess,Genage,Aging,NCBI_ID,Quick_GO,Autophagy-related 8a,obsolete aging,Drosophila melanogaster
2,bchs,Gene_Inhibits_BiologicalProcess,GO:0007568,Gene,None,BiologicalProcess,Genage,Aging,NCBI_ID,Quick_GO,blue cheese,obsolete aging,Drosophila melanogaster
3,car,Gene_Inhibits_BiologicalProcess,GO:0007568,Gene,None,BiologicalProcess,Genage,Aging,NCBI_ID,Quick_GO,carnation,obsolete aging,Drosophila melanogaster
4,Cat,Gene_Inhibits_BiologicalProcess,GO:0007568,Gene,None,BiologicalProcess,Genage,Aging,NCBI_ID,Quick_GO,Catalase,obsolete aging,Drosophila melanogaster
...,...,...,...,...,...,...,...,...,...,...,...,...,...
104,Prp19,Gene_Inhibits_BiologicalProcess,GO:0007568,Gene,None,BiologicalProcess,Genage,Aging,NCBI_ID,Quick_GO,Pre-RNA processing factor 19,obsolete aging,Drosophila melanogaster
105,Sirt4,Gene_Inhibits_BiologicalProcess,GO:0007568,Gene,None,BiologicalProcess,Genage,Aging,NCBI_ID,Quick_GO,Sirtuin 4,obsolete aging,Drosophila melanogaster
106,fkh,Gene_Inhibits_BiologicalProcess,GO:0007568,Gene,None,BiologicalProcess,Genage,Aging,NCBI_ID,Quick_GO,fork head,obsolete aging,Drosophila melanogaster
107,SOD1,Gene_Inhibits_BiologicalProcess,GO:0007568,Gene,None,BiologicalProcess,AgingBank,Aging,NCBI_ID,Quick_GO,"superoxide dismutase 1, soluble",aging,Drosophila melanogaster


# Sanity Check — Distinct Values

In [9]:
for col in ['relation', 'head_type', 'tail_type', 'relation_type', 'kg_source', 'head_id_is', 'tail_id_is']:
    print(f"{col:20s}: {set(final_df[col])}")

relation            : {'Gene_Inhibits_BiologicalProcess'}
head_type           : {'Gene'}
tail_type           : {'BiologicalProcess'}
relation_type       : {None}
kg_source           : {'AgingBank', 'Genage'}
head_id_is          : {'NCBI_ID'}
tail_id_is          : {'Quick_GO'}


In [10]:
# Step 4: drop unresolvable rows
before = len(final_df)
final_df = final_df[~final_df['tail_detail_name'].isna()].reset_index(drop=True)
print(f"Dropped {before - len(final_df):,} unresolvable rows → {len(final_df):,} remaining")

Dropped 0 unresolvable rows → 109 remaining


# NaN Audit (pre-dedup)

In [11]:
true_nan   = final_df.isna().sum()
string_nan = final_df.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_string_count": string_nan,
    'Total_NaN_like':     true_nan + string_nan
})

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,109,0,109
tail_type,0,0,0
kg_source,0,0,0
kg_type,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0


# Deduplication

In [12]:
def merge_sources(x):
    """Combine unique, non-null source labels with '::' delimiter."""
    return '::'.join(sorted(set(x.dropna())))

group_cols = ['head', 'relation', 'tail']

final_df_dedup = final_df.groupby(group_cols, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':        merge_sources,
    'kg_type':          merge_sources,   # ← changed from 'first'
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'species': 'first'
})

print(f"Before dedup: {len(final_df):,}  |  After dedup: {len(final_df_dedup):,}")
final_df_dedup.head(3)

Before dedup: 109  |  After dedup: 109


,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species
0,Atg1,Gene_Inhibits_BiologicalProcess,GO:0007568,Gene,None,BiologicalProcess,Genage,Aging,NCBI_ID,Quick_GO,Autophagy-related 1,obsolete aging,Drosophila melanogaster
1,Atg2,Gene_Inhibits_BiologicalProcess,GO:0007568,Gene,None,BiologicalProcess,Genage,Aging,NCBI_ID,Quick_GO,Autophagy-related 2,obsolete aging,Drosophila melanogaster
2,Atg7,Gene_Inhibits_BiologicalProcess,GO:0007568,Gene,None,BiologicalProcess,Genage,Aging,NCBI_ID,Quick_GO,Autophagy-related 7,obsolete aging,Drosophila melanogaster


In [13]:
true_nan   = final_df_dedup.isna().sum()
string_nan = final_df_dedup.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_string_count": string_nan,
    'Total_NaN_like':     true_nan + string_nan
})

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,109,0,109
tail_type,0,0,0
kg_source,0,0,0
kg_type,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0


In [14]:
print("kg_source values present:", set(final_df_dedup['kg_source']), final_df_dedup['kg_source'].value_counts())

kg_source values present: {'AgingBank', 'Genage'} kg_source
Genage       107
AgingBank      2
Name: count, dtype: int64


In [15]:
print("kg_source values present:", set(final_df_dedup['kg_type']), final_df_dedup['kg_type'].value_counts())

kg_source values present: {'Aging'} kg_type
Aging    109
Name: count, dtype: int64


In [16]:
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 109 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Drosophila/Drosophila_gene_inhibits_biologicalprocess/Drosophila_gene_inhibits_biologicalprocess.csv


In [17]:
# OUT_PATH

In [18]:
# !mkdir Drosophila_gene_inhibits_biologicalprocess